# Empirical Application: College Distance

This notebook replicates the College Distance application using the PLIV DML-IV estimator. The data are downloaded from the AER `CollegeDistance` dataset. We estimate the treatment effect of education on wages using OLS, TSLS, and the DML-IV estimator with Ridge learners.


In [ ]:
import numpy as np
import pandas as pd
import os
from sklearn.model_selection import KFold, GridSearchCV
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import statsmodels.api as sm
from scipy.stats import f as f_dist

# Baseline estimators

def ols_beta_se(Y, D, W):
    n = len(Y)
    X = np.column_stack([D, np.ones(n), W])
    beta = np.linalg.lstsq(X, Y, rcond=None)[0]
    tau = float(beta[0])
    resid = Y - X @ beta
    sigma2 = float((resid.T @ resid)/(n - X.shape[1]))
    var = sigma2 * np.linalg.pinv(X.T @ X)
    se_tau = float(np.sqrt(var[0,0]))
    return tau, se_tau


def tsls_beta_se(Y, D, Z, W):
    n = len(Y)
    Wc = np.column_stack([np.ones(n), W])
    X = np.column_stack([D, Wc])
    Zmat = np.column_stack([Z, Wc])

    ZZ_inv = np.linalg.pinv(Zmat.T @ Zmat)
    PzY = Zmat @ (ZZ_inv @ (Zmat.T @ Y))
    PzX = Zmat @ (ZZ_inv @ (Zmat.T @ X))

    beta = np.linalg.pinv(X.T @ PzX) @ (X.T @ PzY)
    tau = float(beta[0])

    resid = Y - X @ beta
    sigma2 = float((resid.T @ resid) / (n - X.shape[1]))
    var_beta = sigma2 * np.linalg.pinv(X.T @ PzX)
    se_tau = float(np.sqrt(var_beta[0,0]))
    return tau, se_tau


def first_stage_diagnostics(D, Z, W):
    X = sm.add_constant(np.column_stack([Z, W]), has_constant='add')
    Xr = sm.add_constant(W, has_constant='add')
    m_u = sm.OLS(D, X).fit()
    m_r = sm.OLS(D, Xr).fit()

    ssr_u = float(m_u.ssr)
    ssr_r = float(m_r.ssr)

    q = Z.shape[1]
    n = len(D)
    k_u = X.shape[1]

    F = ((ssr_r - ssr_u) / q) / (ssr_u / (n - k_u))
    partial_R2 = (ssr_r - ssr_u) / ssr_r
    pval = float(1 - f_dist.cdf(F, q, n - k_u))
    return {'F': float(F), 'partial_R2': float(partial_R2), 'pval': pval, 'n': int(n), 'dz': int(q)}

# DML-IV Ridge estimator

def fit_ridge_with_cv(X_train, y_train, inner_splits=3, seed=42):
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('model', Ridge())
    ])
    grid = {'model__alpha': np.logspace(-2, 2, 3)}
    inner_cv = KFold(n_splits=inner_splits, shuffle=True, random_state=seed)
    gs = GridSearchCV(pipe, grid, cv=inner_cv, scoring='neg_mean_squared_error', n_jobs=1)
    gs.fit(X_train, y_train)
    return gs.best_estimator_


def pliv_tau_se(Y, D, Z, mu_hat, r_hat, pi_hat):
    n = len(Y)
    Y_tilde = Y - mu_hat
    D_tilde = D - r_hat
    Z_tilde = Z - pi_hat

    Omega = (Z_tilde.T @ Z_tilde) / n
    g_zd  = (Z_tilde.T @ D_tilde) / n
    g_zy  = (Z_tilde.T @ Y_tilde) / n

    Omega_inv = np.linalg.pinv(Omega)

    denom = float(g_zd.T @ Omega_inv @ g_zd)
    numer = float(g_zd.T @ Omega_inv @ g_zy)
    tau_hat = numer / denom

    u_hat = Y_tilde - tau_hat * D_tilde
    psi = Z_tilde * u_hat[:, None]
    Sigma = (psi.T @ psi) / n

    middle = float(g_zd.T @ Omega_inv @ Sigma @ Omega_inv @ g_zd)
    V = (1.0 / denom) * middle * (1.0 / denom)
    se = float(np.sqrt(V / n))
    ci = (tau_hat - 1.96*se, tau_hat + 1.96*se)
    return float(tau_hat), float(se), ci


def dml_iv_ridge(Y, D, Z, W, k_outer=3, k_inner=3, seed=123):
    n, dz = Z.shape
    kf = KFold(n_splits=k_outer, shuffle=True, random_state=seed)

    mu_hat = np.zeros(n)
    r_hat  = np.zeros(n)
    pi_hat = np.zeros((n, dz))

    for fold, (tr, te) in enumerate(kf.split(W)):
        W_tr, W_te = W[tr], W[te]

        mu = fit_ridge_with_cv(W_tr, Y[tr], inner_splits=k_inner, seed=seed+1000+fold)
        r  = fit_ridge_with_cv(W_tr, D[tr], inner_splits=k_inner, seed=seed+2000+fold)
        mu_hat[te] = mu.predict(W_te)
        r_hat[te]  = r.predict(W_te)

        for j in range(dz):
            pj = fit_ridge_with_cv(W_tr, Z[tr, j], inner_splits=k_inner, seed=seed+3000+37*j+fold)
            pi_hat[te, j] = pj.predict(W_te)

    return pliv_tau_se(Y, D, Z, mu_hat, r_hat, pi_hat)

# Data loader for CollegeDistance

def load_college_distance():
    """Load the CollegeDistance dataset from AER (Rdatasets). Returns (Y, D, Z, W_base, W_hd)."""
    url_cd = 'https://raw.githubusercontent.com/vincentarelbundock/Rdatasets/master/csv/AER/CollegeDistance.csv'
    cd = pd.read_csv(url_cd).drop(columns=['rownames'], errors='ignore')

    # Clean data
    cd = cd.loc[cd['wage'].astype(float) > 0].copy()
    cat_cols = ['gender', 'ethnicity', 'fcollege', 'mcollege', 'home', 'urban', 'income', 'region']
    num_cols = ['tuition', 'unemp']
    cd = cd.dropna(subset=['wage', 'education', 'distance'] + cat_cols + num_cols)

    Y = np.log(cd['wage'].astype(float).to_numpy())
    D = cd['education'].astype(float).to_numpy()
    Z = cd[['distance']].astype(float).to_numpy()

    # Base controls: demographic and environmental variables
    W_cat = pd.get_dummies(cd[cat_cols], drop_first=True)
    W_num = cd[num_cols]
    W_base = np.column_stack([W_cat.to_numpy(), W_num.to_numpy()])

    # High-dimensional controls: also include all interactions with distance
    W_hd = np.column_stack([W_base, Z, W_base * Z])
    return Y, D, Z, W_base, W_hd

# Load data and compute estimates

Y, D, Z, W_base, W_hd = load_college_distance()

# First-stage diagnostics (base controls)
fs = first_stage_diagnostics(D, Z, W_base)
print('CollegeDistance first-stage diagnostics:', fs)

# OLS estimate (base controls)
ols_tau, ols_se = ols_beta_se(Y, D, W_base)
print('CollegeDistance OLS:', ols_tau, ols_se)

# TSLS estimate (base controls)
tsls_tau, tsls_se = tsls_beta_se(Y, D, Z, W_base)
print('CollegeDistance TSLS:', tsls_tau, tsls_se)

# DML-IV Ridge estimate (base controls)
dml_tau, dml_se, dml_ci = dml_iv_ridge(Y, D, Z, W_base)
print('CollegeDistance DML-IV (Ridge):', dml_tau, dml_se, dml_ci)

